In [1]:
import os
import pandas as pd
import numpy as np
import psycopg2
import config

from datetime import datetime
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

In [2]:
# --- CREATE CONNECTION ---
engine = create_engine(
    f"postgresql+psycopg2://{config.DB_USER}:{config.DB_PASSWORD}@{config.DB_HOST}:{config.DB_PORT}/{config.DB_NAME}"
)

In [ ]:
LOG_TABLE = "ingestion_wildlife_strike_log"
STG_TABLE = "stg_wildlife_strike"



SQL_GET_PENDING_BATCHES = text(f"""
    SELECT batch_id
    FROM {LOG_TABLE}
    WHERE process_status IN ('PROCESSED', 'FAILED_ANALYTIC')
    ORDER BY ingestion_timestamp, batch_id
""")

SQL_MARK_ANALYTIC_RUNNING = text(f"""
    UPDATE {LOG_TABLE}
    SET process_status = 'ANALYTIC_RUNNING'
    WHERE batch_id = :batch_id
""")

SQL_MARK_ANALYTIC_DONE = text(f"""
    UPDATE {LOG_TABLE}
    SET process_status = 'ANALYTIC'
    WHERE batch_id = :batch_id
""")

SQL_MARK_ANALYTIC_FAILED = text(f"""
    UPDATE {LOG_TABLE}
    SET process_status = 'FAILED_ANALYTIC'
    WHERE batch_id = :batch_id
""")


# -----------------------------
# DIM DATE
# -----------------------------
SQL_INSERT_DIM_DATE = text(f"""
    INSERT INTO dim_date (
        full_date,
        year,
        month,
        day,
        day_of_week,
        month_name,
        quarter,
        is_weekend
    )
    SELECT DISTINCT
        make_date(s.incident_year, s.incident_month, s.incident_day) AS full_date,
        s.incident_year AS year,
        s.incident_month AS month,
        s.incident_day AS day,
        EXTRACT(DOW FROM make_date(s.incident_year, s.incident_month, s.incident_day))::int AS day_of_week,
        TRIM(TO_CHAR(make_date(s.incident_year, s.incident_month, s.incident_day), 'Month')) AS month_name,
        EXTRACT(QUARTER FROM make_date(s.incident_year, s.incident_month, s.incident_day))::int AS quarter,
        CASE
            WHEN EXTRACT(DOW FROM make_date(s.incident_year, s.incident_month, s.incident_day)) IN (0, 6)
            THEN TRUE ELSE FALSE
        END AS is_weekend
    FROM {STG_TABLE} s
    WHERE s.batch_id = :batch_id
      AND s.incident_year IS NOT NULL
      AND s.incident_month IS NOT NULL
      AND s.incident_day IS NOT NULL
      AND NOT EXISTS (
          SELECT 1
          FROM dim_date d
          WHERE d.full_date = make_date(s.incident_year, s.incident_month, s.incident_day)
      )
""")


# -----------------------------
# DIM AIRPORT
# -----------------------------
SQL_INSERT_DIM_AIRPORT = text(f"""
    INSERT INTO dim_airport (
        airport_id,
        airport_name,
        state,
        faa_region
    )
    SELECT DISTINCT
        s.airport_id,
        s.airport_name,
        s.state,
        s.faa_region
    FROM {STG_TABLE} s
    WHERE s.batch_id = :batch_id
      AND NOT EXISTS (
      SELECT 1
      FROM dim_airport d
      WHERE d.airport_id = s.airport_id
        AND d.airport_name = s.airport_name
        AND d.state = s.state
        AND d.faa_region = s.faa_region
      )
""")


# -----------------------------
# DIM OPERATOR
# -----------------------------
SQL_INSERT_DIM_OPERATOR = text(f"""
    INSERT INTO dim_operator (
        operator_id,
        operator_name
    )
    SELECT DISTINCT
        s.operator_id,
        s.operator_name
    FROM {STG_TABLE} s
    WHERE s.batch_id = :batch_id
      AND NOT EXISTS (
          SELECT 1
          FROM dim_operator d
          WHERE d.operator_id = s.operator_id
          AND d.operator_name = s.operator_name
      )
""")


# -----------------------------
# DIM AIRCRAFT
# -----------------------------
SQL_INSERT_DIM_AIRCRAFT = text(f"""
    INSERT INTO dim_aircraft (
        aircraft,
        aircraft_type,
        aircraft_make,
        aircraft_model,
        aircraft_mass,
        engine_make,
        engine_model,
        engines,
        engine_type
    )
    SELECT DISTINCT
        s.aircraft,
        s.aircraft_type,
        s.aircraft_make,
        s.aircraft_model,
        s.aircraft_mass,
        s.engine_make,
        s.engine_model,
        s.engines,
        s.engine_type
    FROM {STG_TABLE} s
    WHERE s.batch_id = :batch_id
      AND NOT EXISTS (
          SELECT 1
          FROM dim_aircraft d
          WHERE d.aircraft = s.aircraft
            AND d.aircraft_type = s.aircraft_type
            AND d.aircraft_make = s.aircraft_make
            AND d.aircraft_model = s.aircraft_model
      )
""")



# -----------------------------
# DIM SPECIES
# -----------------------------
SQL_INSERT_DIM_SPECIES = text(f"""
    INSERT INTO dim_species (
    species_id,
    species_name
)
SELECT DISTINCT
    s.species_id,
    s.species_name
FROM stg_wildlife_strike s
WHERE s.batch_id = :batch_id
  AND NOT EXISTS (
          SELECT 1
          FROM dim_species d
          WHERE d.species_id = s.species_id
            AND d.species_name = s.species_name
      )
""")


# -----------------------------
# DIM VISIBILITY
# -----------------------------
SQL_INSERT_DIM_VISIBILITY = text(f"""
    INSERT INTO dim_visibility (
    visibility
)
SELECT DISTINCT
    s.visibility
FROM stg_wildlife_strike s
WHERE s.batch_id = :batch_id
  AND NOT EXISTS (
          SELECT 1
          FROM dim_visibility d
          WHERE d.visibility = s.visibility
      )
""")



# -----------------------------
# DIM PRECIPITATION
# -----------------------------
SQL_INSERT_DIM_PRECIPITATION = text(f"""
    INSERT INTO dim_precipitation (
    precipitation
)
SELECT DISTINCT
    s.precipitation
FROM stg_wildlife_strike s
WHERE s.batch_id = :batch_id
  AND NOT EXISTS (
          SELECT 1
          FROM dim_precipitation d
          WHERE d.precipitation = s.precipitation
      )
""")


# -----------------------------
# DIM FLIGHT_PHASE
# -----------------------------
SQL_INSERT_DIM_FLIGHT_PHASE = text(f"""
    INSERT INTO dim_flight_phase (
    flight_phase
)
SELECT DISTINCT
    s.flight_phase
FROM stg_wildlife_strike s
WHERE s.batch_id = :batch_id
  AND NOT EXISTS (
          SELECT 1
          FROM dim_flight_phase d
          WHERE d.flight_phase = s.flight_phase
      )
""")


# -----------------------------
# DIM WARNING
# -----------------------------
SQL_INSERT_DIM_WARNING = text(f"""
    INSERT INTO dim_warning (
    warning_issued
)
SELECT DISTINCT
    s.warning_issued
FROM stg_wildlife_strike s
WHERE s.batch_id = :batch_id
  AND NOT EXISTS (
          SELECT 1
          FROM dim_warning d
          WHERE d.warning_issued = s.warning_issued
      )
""")


# -----------------------------
# DIM FLIGHT_IMPACT
# -----------------------------
SQL_INSERT_DIM_FLIGHT_IMPACT = text(f"""
    INSERT INTO dim_flight_impact (
    flight_impact
)
SELECT DISTINCT
    s.flight_impact
FROM stg_wildlife_strike s
WHERE s.batch_id = :batch_id
  AND NOT EXISTS (
          SELECT 1
          FROM dim_flight_impact d
          WHERE d.flight_impact = s.flight_impact
      )
""")


# -----------------------------
# DIM SPECIES_QUANTITY
# -----------------------------
SQL_INSERT_DIM_SPECIES_QUANTITY = text(f"""
    INSERT INTO dim_species_quantity (
    species_quantity
)
SELECT DISTINCT
    s.species_quantity
FROM stg_wildlife_strike s
WHERE s.batch_id = :batch_id
  AND NOT EXISTS (
          SELECT 1
          FROM dim_species_quantity d
          WHERE d.species_quantity = s.species_quantity
      )
""")




# -----------------------------
# FACT TABLE
# -----------------------------
# Adjust columns here based on your final fact table design.
SQL_INSERT_FACT = text(f"""
    INSERT INTO fact_wildlife_strike (
    date_id,
    airport_id,
    operator_id,
    aircraft_id,
    species_id,
    visibility_id,
    precipitation_id,
    flight_phase_id,
    warning_id,
    flight_impact_id,
    species_quantity_id,

    record_id,

    height,
    speed,
    distance,
    fatalities,
    injuries,

    aircraft_damage,
    radome_strike,
    radome_damage,
    windshield_strike,
    windshield_damage,
    nose_strike,
    nose_damage,
    engine1_strike,
    engine1_damage,
    engine2_strike,
    engine2_damage,
    engine3_strike,
    engine3_damage,
    engine4_strike,
    engine4_damage,
    engine_ingested,
    propeller_strike,
    propeller_damage,
    wing_or_rotor_strike,
    wing_or_rotor_damage,
    fuselage_strike,
    fuselage_damage,
    landing_gear_strike,
    landing_gear_damage,
    tail_strike,
    tail_damage,
    lights_strike,
    lights_damage,
    other_strike,
    other_damage
)
SELECT
    -- DIMENSIONS (SURROGATE KEYS)
    d_date.id,
    d_airport.id,
    d_operator.id,
    d_aircraft.id,
    d_species.id,
    d_visibility.id,
    d_precipitation.id,
    d_phase.id,
    d_warning.id,
    d_impact.id,
    d_species_qty.id,

    -- FACT ATTRIBUTES
    s.record_id,

    s.height,
    s.speed,
    s.distance,
    s.fatalities,
    s.injuries,

    s.aircraft_damage,
    s.radome_strike,
    s.radome_damage,
    s.windshield_strike,
    s.windshield_damage,
    s.nose_strike,
    s.nose_damage,
    s.engine1_strike,
    s.engine1_damage,
    s.engine2_strike,
    s.engine2_damage,
    s.engine3_strike,
    s.engine3_damage,
    s.engine4_strike,
    s.engine4_damage,
    s.engine_ingested,
    s.propeller_strike,
    s.propeller_damage,
    s.wing_or_rotor_strike,
    s.wing_or_rotor_damage,
    s.fuselage_strike,
    s.fuselage_damage,
    s.landing_gear_strike,
    s.landing_gear_damage,
    s.tail_strike,
    s.tail_damage,
    s.lights_strike,
    s.lights_damage,
    s.other_strike,
    s.other_damage

FROM stg_wildlife_strike s

-- DATE
JOIN dim_date d_date
    ON d_date.full_date = make_date(
        s.incident_year,
        s.incident_month,
        s.incident_day
    )

-- AIRPORT
JOIN dim_airport d_airport
    ON d_airport.airport_id = s.airport_id
   AND d_airport.airport_name = s.airport_name
   AND d_airport.state = s.state
   AND d_airport.faa_region = s.faa_region

-- OPERATOR
JOIN dim_operator d_operator
    ON d_operator.operator_id = s.operator_id
   AND d_operator.operator_name = s.operator_name

-- AIRCRAFT
JOIN dim_aircraft d_aircraft
    ON d_aircraft.aircraft = s.aircraft
   AND d_aircraft.aircraft_type = s.aircraft_type
   AND d_aircraft.aircraft_make = s.aircraft_make
   AND d_aircraft.aircraft_model = s.aircraft_model
   AND COALESCE(d_aircraft.aircraft_mass, -1) = COALESCE(s.aircraft_mass, -1)
   AND d_aircraft.engine_make = s.engine_make
   AND d_aircraft.engine_model = s.engine_model
   AND COALESCE(d_aircraft.engines, -1) = COALESCE(s.engines, -1)
   AND d_aircraft.engine_type = s.engine_type

-- SPECIES
JOIN dim_species d_species
    ON d_species.species_id = s.species_id
   AND d_species.species_name = s.species_name

-- SIMPLE DIMENSIONS
JOIN dim_visibility d_visibility
    ON d_visibility.visibility = s.visibility

JOIN dim_precipitation d_precipitation
    ON d_precipitation.precipitation = s.precipitation

JOIN dim_flight_phase d_phase
    ON d_phase.flight_phase = s.flight_phase

JOIN dim_warning d_warning
    ON d_warning.warning_issued = s.warning_issued

JOIN dim_flight_impact d_impact
    ON d_impact.flight_impact = s.flight_impact

JOIN dim_species_quantity d_species_qty
    ON d_species_qty.species_quantity = s.species_quantity

-- INCREMENTAL LOAD
WHERE s.batch_id = :batch_id

-- AVOID DUPLICATE FACT
AND NOT EXISTS (
    SELECT 1
    FROM fact_wildlife_strike f
    WHERE f.record_id = s.record_id
);
""")



In [4]:

# =========================================================
# ETL FUNCTIONS
# =========================================================

def get_pending_batches():
    with engine.connect() as conn:
        rows = conn.execute(SQL_GET_PENDING_BATCHES).fetchall()
        return [row[0] for row in rows]


def load_dimensions(conn, batch_id: str):
    print(f"[{batch_id}] Loading dim_date...")
    conn.execute(SQL_INSERT_DIM_DATE, {"batch_id": batch_id})

    print(f"[{batch_id}] Loading dim_airport...")
    conn.execute(SQL_INSERT_DIM_AIRPORT, {"batch_id": batch_id})

    print(f"[{batch_id}] Loading dim_operator...")
    conn.execute(SQL_INSERT_DIM_OPERATOR, {"batch_id": batch_id})

    print(f"[{batch_id}] Loading dim_aircraft...")
    conn.execute(SQL_INSERT_DIM_AIRCRAFT, {"batch_id": batch_id})

    print(f"[{batch_id}] Loading dim_species...")
    conn.execute(SQL_INSERT_DIM_SPECIES, {"batch_id": batch_id})

    print(f"[{batch_id}] Loading dim_visibility...")
    conn.execute(SQL_INSERT_DIM_VISIBILITY, {"batch_id": batch_id})
    
    print(f"[{batch_id}] Loading dim_precipitation...")
    conn.execute(SQL_INSERT_DIM_PRECIPITATION, {"batch_id": batch_id})

    print(f"[{batch_id}] Loading dim_flight_phase...")
    conn.execute(SQL_INSERT_DIM_FLIGHT_PHASE, {"batch_id": batch_id})

    print(f"[{batch_id}] Loading dim_warning...")
    conn.execute(SQL_INSERT_DIM_WARNING, {"batch_id": batch_id})

    print(f"[{batch_id}] Loading dim_flight_impact...")
    conn.execute(SQL_INSERT_DIM_FLIGHT_IMPACT, {"batch_id": batch_id})

    print(f"[{batch_id}] Loading dim_species_quantity...")
    conn.execute(SQL_INSERT_DIM_SPECIES_QUANTITY, {"batch_id": batch_id})


def load_fact(conn, batch_id: str):
    print(f"[{batch_id}] Loading fact_wildlife_strike...")
    conn.execute(SQL_INSERT_FACT, {"batch_id": batch_id})


def process_batch(batch_id: str):
    print(f"\n=== Start batch {batch_id} ===")

    try:
        with engine.begin() as conn:
            conn.execute(SQL_MARK_ANALYTIC_RUNNING, {"batch_id": batch_id})
            load_dimensions(conn, batch_id)
            load_fact(conn, batch_id)
            conn.execute(SQL_MARK_ANALYTIC_DONE, {"batch_id": batch_id})

        print(f"=== Batch {batch_id} completed ===")

    except Exception as e:
        print(f"!!! Batch {batch_id} failed: {e}")

        with engine.begin() as conn:
            conn.execute(SQL_MARK_ANALYTIC_FAILED, {"batch_id": batch_id})

        raise


def ingest_analytics():
    print("Searching pending analytic batches...")
    batch_ids = get_pending_batches()

    if not batch_ids:
        print("No batch to process.")
        return

    print(f"Found {len(batch_ids)} batch(es).")

    for batch_id in batch_ids:
        try:
            process_batch(batch_id)
        except Exception:
            # continue next batch
            continue




In [5]:
ingest_analytics()

Searching pending analytic batches...
Found 1 batch(es).

=== Start batch 20260320211827 ===
[20260320211827] Loading dim_date...
[20260320211827] Loading dim_airport...
[20260320211827] Loading dim_operator...
[20260320211827] Loading dim_aircraft...
[20260320211827] Loading dim_species...
[20260320211827] Loading dim_visibility...
[20260320211827] Loading dim_precipitation...
[20260320211827] Loading dim_flight_phase...
[20260320211827] Loading dim_warning...
[20260320211827] Loading dim_flight_impact...
[20260320211827] Loading dim_species_quantity...
[20260320211827] Loading fact_wildlife_strike...
=== Batch 20260320211827 completed ===
